# MA3001B • Semana 08: Suavizado Exponencial y Benchmark Integrador
## Holt-Winters Amortiguado, Validación con Origen Móvil y Torneo de 5 Modelos
**Profesor B: Modelos y TDA • Tecnológico de Monterrey**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alxcn/Models_MA3001B/blob/main/notebooks/N5_HoltWinters_Benchmark.ipynb)

---
### Objetivos de la Sesión
1. Formular y ajustar **Suavizado Exponencial de Holt-Winters** con tendencia amortiguada y estacionalidad aditiva.
2. Implementar un protocolo de evaluación riguroso con **5 orígenes móviles consecutivos** (rolling origin cross-validation).
3. Computar la métrica estandarizada e independiente de escala **MASE** (Mean Absolute Scaled Error).
4. Ejecutar el **benchmark final** comparando 5 familias: Ingenuo Estacional, Holt-Winters, SARIMA, LSTM y Transformer.
5. Elaborar una recomendación técnica justificada para producción respondiendo: *¿Superó el deep learning a los modelos estadísticos clásicos?*


## 1. Problema Detonador: La Fragilidad del Corte Único
Un solo corte de prueba puede favorecer a una arquitectura por azar estadístico. Veamos cómo un torneo multicorte con origen móvil revela la verdadera consistencia de cada modelo.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

y_det = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()

# Comparación rápida en un solo corte:
tr_1, te_1 = y_det.iloc[:-12], y_det.iloc[-12:]
hw_1 = ExponentialSmoothing(tr_1, trend="add", damped_trend=True, seasonal="add", seasonal_periods=12, initialization_method="estimated").fit()
fc_hw1 = hw_1.forecast(12).values
mae_hw1 = np.mean(np.abs(te_1.values - fc_hw1))

print(f"MAE Holt-Winters en el último año: {mae_hw1:.4f} ppm")
print("-> ¿Se mantendrá este orden de rendimiento si evaluamos a lo largo de 5 años distintos?")


## 2. Marco Teórico Mínimo

### 2.1 Ecuaciones de Holt-Winters con Tendencia Amortiguada (Aditivo)
$$\begin{aligned}
\ell_t &= \alpha (y_t - s_{t-m}) + (1 - \alpha)(\ell_{t-1} + \phi b_{t-1}) \\
b_t &= \beta (\ell_t - \ell_{t-1}) + (1 - \beta) \phi b_{t-1} \\
s_t &= \gamma (y_t - \ell_{t-1} - \phi b_{t-1}) + (1 - \gamma) s_{t-m} \\
\hat{y}_{t+h|t} &= \ell_t + \left(\sum_{i=1}^h \phi^i\right) b_t + s_{t+h-m(k+1)}
\end{aligned}$$

### 2.2 Métrica MASE (Mean Absolute Scaled Error)
$$\operatorname{MASE} = \frac{\frac{1}{h} \sum_{j=1}^h |y_{T+j} - \hat{y}_{T+j}|}{\frac{1}{T-m} \sum_{t=m+1}^T |y_t - y_{t-m}|}$$


### Configuración de Datos: Práctica (CO₂) o Serie del Reto
En la siguiente celda se carga la serie de práctica por defecto (**CO₂ mensual de Mauna Loa** de `statsmodels`).
Para aplicar el análisis a la serie de su equipo en el Reto, cambie la bandera `USAR_SERIE_RETO = True` y proporcione la ruta a su archivo CSV.

In [ ]:
# ==============================================================================
# CELDA SERIE_RETO: Alternar entre datos de práctica y datos del Reto
# ==============================================================================
USAR_SERIE_RETO = False  # Cambiar a True para usar la serie de su Reto

if USAR_SERIE_RETO:
    # Sustituya con la carga de la serie de su equipo:
    # df_reto = pd.read_csv("datos_reto.csv", index_col=0, parse_dates=True)
    # y = df_reto["columna_objetivo"].asfreq("MS").interpolate()
    print("Modo Reto activado.")
else:
    # Serie de práctica por defecto: CO2 de Mauna Loa (mensual interpolado)
    y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
    print(f"Cargada serie de práctica CO2: {len(y)} observaciones ({y.index.min().strftime('%Y-%m')} a {y.index.max().strftime('%Y-%m')})")


## 3. Ejemplo Aplicado Paso a Paso

### Paso 1: Ajuste de Holt-Winters y Extracción de Parámetros

In [ ]:
hw_fit = ExponentialSmoothing(
    y.iloc[:-24],
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
).fit()

print("=== PARÁMETROS ESTIMADOS HOLT-WINTERS ===")
for p_name in ["smoothing_level", "smoothing_trend", "smoothing_seasonal", "damping_trend"]:
    print(f"{p_name:<22}: {hw_fit.params.get(p_name, np.nan):.4f}")
print(f"AIC del modelo HW: {hw_fit.aic:.2f}")


### Paso 2: Protocolo de Evaluación con 5 Orígenes Móviles (h = 12 meses)

In [ ]:
h = 12
n_origenes = 5
# Generamos 5 puntos de corte cronológicos consecutivos
origenes = [len(y) - (k + 1) * h for k in reversed(range(n_origenes))]

print(f"Orígenes temporales de evaluación (índices): {origenes}")
print(f"Fechas de corte de pronóstico: {[y.index[o].strftime('%Y-%m') for o in origenes]}")


### Paso 3: Funciones de Pronóstico de los Cinco Modelos

In [ ]:
def f_naive_estacional(tr, h, m=12):
    return np.tile(tr.values[-m:], int(np.ceil(h / m)))[:h]

def f_hw(tr, h):
    aj = ExponentialSmoothing(tr, trend="add", damped_trend=True, seasonal="add",
                              seasonal_periods=12, initialization_method="estimated").fit()
    return aj.forecast(h).values

def f_sarima(tr, h):
    aj = SARIMAX(tr, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    return aj.forecast(h).values

def make_windows(z, L):
    X = np.stack([z[i:i + L] for i in range(len(z) - L)])
    t = z[L:]
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(t, dtype=torch.float32).unsqueeze(-1)

class SimpleLSTM(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

def f_lstm(tr, h, L=24, epochs=25):
    sc = StandardScaler().fit(tr.to_frame())
    z = sc.transform(tr.to_frame()).ravel().astype("float32")
    X, t = make_windows(z, L)
    dl = DataLoader(TensorDataset(X, t), batch_size=32, shuffle=True)
    
    m = SimpleLSTM()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            opt.zero_grad()
            nn.MSELoss()(m(xb), yb).backward()
            opt.step()
            
    m.eval()
    ventana = list(z[-L:])
    pred_norm = []
    with torch.no_grad():
        for _ in range(h):
            inp = torch.tensor(ventana[-L:], dtype=torch.float32).view(1, L, 1)
            nxt = m(inp).item()
            pred_norm.append(nxt)
            ventana.append(nxt)
    return sc.inverse_transform(np.array(pred_norm).reshape(-1, 1)).ravel()

def mase(real, pred, tr, m=12):
    denom = np.mean(np.abs(tr.values[m:] - tr.values[:-m]))
    if denom == 0:
        return np.nan
    return np.mean(np.abs(real - pred)) / denom


### Paso 4: Ejecución del Torneo y Tabla de Resultados

In [ ]:
modelos = {
    "1. Ingenuo Estacional": f_naive_estacional,
    "2. Holt-Winters": f_hw,
    "3. SARIMA (Airline)": f_sarima,
    "4. LSTM Recurrente": f_lstm
}

filas_bench = []
for nombre, fn_modelo in modelos.items():
    print(f"Evaluando: {nombre}...")
    for idx_o, o in enumerate(origenes):
        tr = y.iloc[:o]
        real = y.iloc[o:o + h].values
        pred = fn_modelo(tr, h)
        
        mae_val = np.mean(np.abs(real - pred))
        rmse_val = np.sqrt(np.mean((real - pred) ** 2))
        mase_val = mase(real, pred, tr)
        
        filas_bench.append({
            "Modelo": nombre,
            "Origen": idx_o + 1,
            "MAE": mae_val,
            "RMSE": rmse_val,
            "MASE": mase_val
        })

df_bench = pd.DataFrame(filas_bench)
resumen_promedio = df_bench.groupby("Modelo")[["MAE", "RMSE", "MASE"]].mean().sort_values("MASE")

print("\n" + "="*55)
print("=== TABLA FINAL DEL BENCHMARK (Promedio 5 Orígenes) ===")
print("="*55)
print(resumen_promedio.round(4))


### Paso 5: Visualización de las Trayectorias de Pronóstico

In [ ]:
# Visualización en el último origen de pronóstico
ultimo_o = origenes[-1]
tr_plot = y.iloc[:ultimo_o]
real_plot = y.iloc[ultimo_o:ultimo_o + h]

plt.figure(figsize=(10, 5))
plt.plot(tr_plot.index[-24:], tr_plot.iloc[-24:], label="Histórico (Train)", color="#073642")
plt.plot(real_plot.index, real_plot, label="Real (Test)", color="black", lw=2.5)

for nombre, fn_modelo in modelos.items():
    pred_vals = fn_modelo(tr_plot, h)
    plt.plot(real_plot.index, pred_vals, linestyle="--", label=nombre, lw=1.8)

plt.title("Comparación de Pronósticos en el Último Origen Móvil (h=12)")
plt.xlabel("Fecha")
plt.ylabel("CO2 (ppm)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## 4. Actividad del Reto y Entregable Final
1. Ejecute el benchmark de 5 orígenes en la serie de su Reto.
2. Complete la tabla comparativa de MAE, RMSE y MASE.
3. Responda con rigor técnico la pregunta clave: *¿Superó el deep learning a los modelos econométricos clásicos?*

In [ ]:
# ==============================================================================
# TODO 1: Torneo en la Serie del Reto
# ==============================================================================
# Ejecute el ciclo de evaluación con su serie y reporte la tabla agrupada.


In [ ]:
# ==============================================================================
# TODO 2: Recomendación Ejecutiva
# ==============================================================================
# Redacte un párrafo justificando qué modelo recomienda desplegar en producción,
# considerando precisión (MASE), intervalos de confianza, coste y facilidad de actualización.


## 5. Conclusiones Globales del Módulo
- **Conclusión 1:** En series de tiempo univariadas con fuerte estructura estacional y tendencia, los modelos clásicos con estructura inductiva adecuada (Holt-Winters, SARIMA) suelen superar o igualar a redes neuronales pesadas con una fracción diminuta del costo computacional.
- **Conclusión 2:** El rol del científico de datos no es aplicar el modelo más complejo por defecto, sino contrastar de forma empírica y reproducible el desempeño frente a baselines sólidos.
